1. Loading data from Drive

In [10]:
from google.colab import drive
drive.mount('/content/drive')

BASE = '/content/drive/MyDrive/mzazicare-ml'
DATA = f'{BASE}/data'

import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')
np.random.seed(42)

df_full = pd.read_parquet(f'{DATA}/checkpoint_all_features.parquet')
print(f"Loaded: {df_full.shape}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Loaded: (1563463, 32)


2. Calibrating severity against real attack distributions

In [11]:
# Extracting real attack behaviour to calibrate how extreme synthetic anomalies should be
real_attacks = df_full[df_full['Is Attack IP'] == 1]
real_normal = df_full[(df_full['Is Attack IP'] == 0) & (df_full['Login Successful'] == 1)]

print("=== REAL ATTACK DISTRIBUTION (used to calibrate synthetic severity) ===")
print(f"\nLogin velocity — Normal mean: {real_normal['login_velocity'].mean():.2f}, "
      f"Attack mean: {real_attacks['login_velocity'].mean():.2f}, "
      f"Attack 95th pct: {real_attacks['login_velocity'].quantile(0.95):.2f}")

print(f"\nTime deviation — Normal mean: {real_normal['time_deviation'].mean():.2f}, "
      f"Attack mean: {real_attacks['time_deviation'].mean():.2f}, "
      f"Attack 95th pct: {real_attacks['time_deviation'].quantile(0.95):.2f}")

foreign_rate_attack = real_attacks['is_foreign_login'].mean()
foreign_rate_normal = real_normal['is_foreign_login'].mean()
print(f"\nForeign login rate — Normal: {foreign_rate_normal:.4f}, Attack: {foreign_rate_attack:.4f}")

# Store calibration reference values
CALIBRATION = {
    'velocity_95th_attack': real_attacks['login_velocity'].quantile(0.95),
    'velocity_max_attack': real_attacks['login_velocity'].quantile(0.99),
    'time_dev_95th_attack': real_attacks['time_deviation'].quantile(0.95),
    'normal_velocity_mean': real_normal['login_velocity'].mean(),
    'normal_velocity_std': real_normal['login_velocity'].std(),
}
print("\nCalibration reference values:")
for k, v in CALIBRATION.items():
    print(f"  {k}: {v:.3f}")

=== REAL ATTACK DISTRIBUTION (used to calibrate synthetic severity) ===

Login velocity — Normal mean: 1.02, Attack mean: 49.76, Attack 95th pct: 172.00

Time deviation — Normal mean: 1.05, Attack mean: 2.81, Attack 95th pct: 9.80

Foreign login rate — Normal: 0.0041, Attack: 0.1366

Calibration reference values:
  velocity_95th_attack: 172.000
  velocity_max_attack: 236.000
  time_dev_95th_attack: 9.797
  normal_velocity_mean: 1.024
  normal_velocity_std: 0.180


3. Buiding per-user baseline profile from real normal users

In [12]:
# Selecting a sample of real users with sufficient login history to build reliable baselines
user_login_counts = real_normal.groupby('User ID').size()
eligible_users = user_login_counts[user_login_counts >= 5].index

print(f"Users with 5+ logins: {len(eligible_users):,}")

# Sample users for the synthetic dataset
N_USERS = min(5000, len(eligible_users))
sampled_users = np.random.choice(eligible_users, size=N_USERS, replace=False)

baseline_df = real_normal[real_normal['User ID'].isin(sampled_users)]

user_profiles = baseline_df.groupby('User ID').agg(
    mean_hour=('login_hour', 'mean'),
    std_hour=('login_hour', 'std'),
    usual_country=('usual_country', 'first'),
    usual_city=('usual_city', 'first'),
    mean_velocity=('login_velocity', 'mean'),
    mean_day=('login_day_of_week', lambda x: x.mode()[0] if not x.mode().empty else 0),
).reset_index()

user_profiles['std_hour'] = user_profiles['std_hour'].fillna(1.0)

print(f"\nBuilt baseline profiles for {len(user_profiles):,} real users")
user_profiles.head()

Users with 5+ logins: 4,924

Built baseline profiles for 4,924 real users


,User ID,mean_hour,std_hour,usual_country,usual_city,mean_velocity,mean_day
0,-9222331807151947514,6.333333,0.516398,NO,-,1.0,0
1,-9222035471022096918,13.714286,5.794086,NO,Oslo,1.0,0
2,-9219055607219276946,10.066667,3.807261,NO,Ovrebo,1.0,1
3,-9217866553275673565,11.000000,6.228965,NO,Oslo,1.0,0
4,-9212178176835340589,6.500000,0.534522,NO,Hamar,1.0,3


4. Generating functions for normal and anomalous records

In [13]:
def generate_normal_record(profile):
    """Generate a plausible normal login close to this user's own real baseline."""
    hour = np.clip(np.random.normal(profile['mean_hour'], max(profile['std_hour'], 0.5)), 0, 23)
    return {
        'User ID': profile['User ID'],
        'login_hour': round(hour),
        'login_day_of_week': profile['mean_day'],
        'time_deviation': abs(round(hour) - profile['mean_hour']),
        'is_known_country': 1,
        'is_known_city': 1,
        'is_foreign_login': 0,
        'location_anomaly_score': 0,
        'is_known_device': 1,
        'login_velocity': max(1, round(np.random.normal(profile['mean_velocity'], 1))),
        'anomaly_type': 'none',
        'severity': 'none',
        'is_anomaly': 0,
    }

def generate_anomalous_record(profile, anomaly_type, severity, calibration):
    """
    Generate an anomaly by perturbing a real user's baseline.
    Severity levels are calibrated against real attack distributions from Cell 2.
    """
    record = generate_normal_record(profile)
    record['anomaly_type'] = anomaly_type
    record['severity'] = severity
    record['is_anomaly'] = 1

    severity_multiplier = {'mild': 0.5, 'moderate': 1.0, 'severe': 1.5}[severity]
    severity_score_map = {'mild': 1.0, 'moderate': 1.5, 'severe': 2.0}

    if anomaly_type == 'time':
        # Push login hour far from the user's own mean, scaled by real attack time_deviation
        deviation = calibration['time_dev_95th_attack'] * severity_multiplier
        direction = np.random.choice([-1, 1])
        new_hour = (profile['mean_hour'] + direction * deviation) % 24
        record['login_hour'] = round(new_hour)
        record['time_deviation'] = deviation

    elif anomaly_type == 'location':
        # Simulate login from an unfamiliar country/city, graded by severity
        if severity == 'mild':
            # Unfamiliar city, same country
            record['is_known_country'] = 1
            record['is_known_city'] = 0
            record['is_foreign_login'] = 0
        elif severity == 'moderate':
            # Foreign country, but a relatively common/plausible one
            record['is_known_country'] = 0
            record['is_known_city'] = 0
            record['is_foreign_login'] = 1
        else:  # severe
            # Foreign country, geographically implausible given recent activity
            record['is_known_country'] = 0
            record['is_known_city'] = 0
            record['is_foreign_login'] = 1

        record['location_anomaly_score'] = severity_score_map[severity]

    elif anomaly_type == 'velocity':
        # Spike login velocity, scaled against real attack velocity distribution
        target = calibration['velocity_95th_attack'] * severity_multiplier
        record['login_velocity'] = max(record['login_velocity'], round(target))

    elif anomaly_type == 'device':
        record['is_known_device'] = 0

    elif anomaly_type == 'combined':
        # Severe case — multiple signals fire simultaneously (most realistic account takeover pattern)
        record['is_known_country'] = 0
        record['is_known_city'] = 0
        record['is_foreign_login'] = 1
        record['location_anomaly_score'] = severity_score_map[severity]
        record['is_known_device'] = 0
        record['login_velocity'] = max(record['login_velocity'],
                                        round(calibration['velocity_95th_attack'] * severity_multiplier))
        deviation = calibration['time_dev_95th_attack'] * severity_multiplier
        record['time_deviation'] = deviation

    return record


print("Generator functions defined")


Generator functions defined


5. Generating the full synthetic dataset

In [14]:
records = []

# 1. Normal records — majority class (one per user baseline, repeated with slight noise)
for _, profile in user_profiles.iterrows():
    for _ in range(3):  # 3 normal samples per user
        records.append(generate_normal_record(profile))

# 2. Anomalous records — across all types and severities
anomaly_types = ['time', 'location', 'velocity', 'device', 'combined']
severities = ['mild', 'moderate', 'severe']

# Sampling a subset of users to receive anomaly injections (keeps overall contamination realistic, about 8-10%)
anomaly_user_sample = user_profiles.sample(n=int(len(user_profiles) * 0.3), random_state=42)

for _, profile in anomaly_user_sample.iterrows():
    anomaly_type = np.random.choice(anomaly_types)
    severity = np.random.choice(severities, p=[0.5, 0.3, 0.2])  # most anomalies are mild/moderate, fewer severe
    records.append(generate_anomalous_record(profile, anomaly_type, severity, CALIBRATION))

synthetic_df = pd.DataFrame(records)

print(f"Synthetic dataset generated: {synthetic_df.shape}")
print(f"\nClass balance:")
print(synthetic_df['is_anomaly'].value_counts())
print(f"Anomaly rate: {synthetic_df['is_anomaly'].mean()*100:.2f}%")
print(f"\nAnomaly type breakdown:")
print(synthetic_df[synthetic_df['is_anomaly']==1]['anomaly_type'].value_counts())
print(f"\nSeverity breakdown:")
print(synthetic_df[synthetic_df['is_anomaly']==1]['severity'].value_counts())

Synthetic dataset generated: (16249, 13)

Class balance:
is_anomaly
0    14772
1     1477
Name: count, dtype: int64
Anomaly rate: 9.09%

Anomaly type breakdown:
anomaly_type
time        325
combined    312
location    288
device      277
velocity    275
Name: count, dtype: int64

Severity breakdown:
severity
mild        740
moderate    428
severe      309
Name: count, dtype: int64


6. Validating feature separability

In [16]:
features_check = ['login_hour', 'time_deviation', 'is_known_country',
                   'is_known_city', 'is_foreign_login', 'location_anomaly_score',
                   'is_known_device', 'login_velocity']

comparison = synthetic_df.groupby('is_anomaly')[features_check].mean().T
comparison.columns = ['Normal', 'Anomaly']
comparison['difference'] = abs(comparison['Anomaly'] - comparison['Normal'])
comparison = comparison.sort_values('difference', ascending=False)
print(comparison.round(4))

                         Normal  Anomaly  difference
login_velocity           1.4113  59.1313     57.7200
time_deviation           2.9866   5.3505      2.3639
location_anomaly_score   0.0000   0.5667      0.5667
is_known_city            1.0000   0.5938      0.4062
is_known_device          1.0000   0.6012      0.3988
is_foreign_login         0.0000   0.3209      0.3209
is_known_country         1.0000   0.6791      0.3209
login_hour              11.9257  11.8246      0.1010


7. Saving the dataset

In [17]:
synthetic_df.to_parquet(f'{DATA}/synthetic_anomaly_dataset.parquet', index=False)
synthetic_df.to_csv(f'{DATA}/synthetic_anomaly_dataset.csv', index=False)
print(f"Saved: {synthetic_df.shape}")
print(f"Columns: {synthetic_df.columns.tolist()}")

Saved: (16249, 13)
Columns: ['User ID', 'login_hour', 'login_day_of_week', 'time_deviation', 'is_known_country', 'is_known_city', 'is_foreign_login', 'location_anomaly_score', 'is_known_device', 'login_velocity', 'anomaly_type', 'severity', 'is_anomaly']
